<a href="https://colab.research.google.com/github/JuanRosales707/LAB_13_Cuellar/blob/develop/LAB_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
productivity_prediction_of_garment_employees = fetch_ucirepo(id=597)

# data (as pandas dataframes)
X = productivity_prediction_of_garment_employees.data.features
y = productivity_prediction_of_garment_employees.data.targets

# metadata
print(productivity_prediction_of_garment_employees.metadata)

# variable information
print(productivity_prediction_of_garment_employees.variables)

{'uci_id': 597, 'name': 'Productivity Prediction of Garment Employees', 'repository_url': 'https://archive.ics.uci.edu/dataset/597/productivity+prediction+of+garment+employees', 'data_url': 'https://archive.ics.uci.edu/static/public/597/data.csv', 'abstract': 'This dataset includes important attributes of the garment manufacturing process and the productivity of the employees which had been collected manually and also been validated by the industry experts.', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 1197, 'num_features': 14, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['actual_productivity'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2020, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C51S6D', 'creators': [], 'intro_paper': {'ID': 399, 'type': 'NATIVE', 'title': 'Mining the productivity dat

In [3]:
# Visualizar las primeras filas de las características (X)
print("Características (X):")
print(X.head())

# Visualizar las primeras filas de los targets (y)
print("\nTargets (y):")
print(y.head())


Características (X):
       date   quarter department       day  team  targeted_productivity  \
0  1/1/2015  Quarter1     sweing  Thursday     8                   0.80   
1  1/1/2015  Quarter1  finishing  Thursday     1                   0.75   
2  1/1/2015  Quarter1     sweing  Thursday    11                   0.80   
3  1/1/2015  Quarter1     sweing  Thursday    12                   0.80   
4  1/1/2015  Quarter1     sweing  Thursday     6                   0.80   

     smv     wip  over_time  incentive  idle_time  idle_men  \
0  26.16  1108.0       7080         98        0.0         0   
1   3.94     NaN        960          0        0.0         0   
2  11.41   968.0       3660         50        0.0         0   
3  11.41   968.0       3660         50        0.0         0   
4  25.90  1170.0       1920         50        0.0         0   

   no_of_style_change  no_of_workers  
0                   0           59.0  
1                   0            8.0  
2                   0           

### **a. Realice el preprocesamiento de la información que incluya el análisis de datos faltantes y tratamiento de outliers a nivel univariado y multivariado. Además, convierta las variables categóricas a dummies y aplique un escalamiento a las variables numéricas.**

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Unir X e y para trabajar con todo el dataset
df = pd.concat([X, y], axis=1)

In [5]:
print("Datos faltantes por columna:")
print(df.isnull().sum())

# Imputar valores nulos con la mediana (solo en columnas numéricas)
df.fillna(df.median(numeric_only=True), inplace=True)

Datos faltantes por columna:
date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64


In [6]:
# Univariado con IQR (boxplot + reemplazo)
def tratar_outliers_iqr(col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < limite_inferior, limite_inferior, df[col])
    df[col] = np.where(df[col] > limite_superior, limite_superior, df[col])

# Aplicar a columnas numéricas
cols_numericas = df.select_dtypes(include=['int64', 'float64']).columns

for col in cols_numericas:
    tratar_outliers_iqr(col)

In [7]:
# Revisar columnas categóricas
cat_cols = df.select_dtypes(include='object').columns
print("\nColumnas categóricas:", list(cat_cols))

# Convertir a variables dummies (one-hot encoding)
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


Columnas categóricas: ['date', 'quarter', 'department', 'day']


In [8]:
# Guardar el nombre de la variable target
target = 'actual_productivity'

# Separar X (sin el target) y y (target)
X_scaled = df.drop(columns=[target])
y_final = df[target]

# Aplicar escalamiento estándar
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_scaled), columns=X_scaled.columns)

# Mostrar resultados
print("\nDatos escalados (primeras filas):")
print(X_scaled.head())

print("\nTarget:")
print(y_final.head())


Datos escalados (primeras filas):
       team  targeted_productivity       smv       wip  over_time  incentive  \
0  0.454323               0.836708  1.014552  0.549326   0.763049   2.259493   
1 -1.567329               0.174314 -1.016778  0.053658  -1.088995  -0.846671   
2  1.320745               0.836708 -0.333878 -0.456377  -0.271917   0.738107   
3  1.609552               0.836708 -0.333878 -0.456377  -0.271917   0.738107   
4 -0.123292               0.836708  0.990783  0.994709  -0.798479   0.738107   

   idle_time  idle_men  no_of_style_change  no_of_workers  ...  \
0        0.0       0.0                 0.0       1.099229  ...   
1        0.0       0.0                 0.0      -1.199268  ...   
2        0.0       0.0                 0.0      -0.185225  ...   
3        0.0       0.0                 0.0      -0.185225  ...   
4        0.0       0.0                 0.0       0.964023  ...   

   quarter_Quarter2  quarter_Quarter3  quarter_Quarter4  quarter_Quarter5  \
0         

### **b. Separe los datos en entrenamiento (80%) y prueba (20%) y entrene los modelos k-NN, SVM,  regresión logística, árbol de clasificación, Random Forest y Naive Bayes definiendo distintos  hiperparámetros para cada modelo.**

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [10]:
# Separar en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_final, test_size=0.2, random_state=42)

# Definir y entrenar modelos con diferentes hiperparámetros

modelos = {
    "k-NN (k=3)": KNeighborsRegressor(n_neighbors=3),
    "k-NN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "SVR (RBF)": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "SVR (Linear)": SVR(kernel='linear', C=0.5),
    "Decision Tree (max_depth=5)": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Decision Tree (max_depth=None)": DecisionTreeRegressor(random_state=42),
    "Random Forest (100 trees)": RandomForestRegressor(n_estimators=100, random_state=42),
    "Random Forest (200 trees)": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
}

In [11]:
# Evaluar modelos
resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    resultados.append({
        "Modelo": nombre,
        "MSE": mse,
        "R²": r2
    })

# Mostrar resultados en un DataFrame ordenado por R²
df_resultados = pd.DataFrame(resultados).sort_values(by="R²", ascending=False)
print(df_resultados)

                           Modelo       MSE        R²
7       Random Forest (200 trees)  0.012164  0.514438
6       Random Forest (100 trees)  0.012411  0.504601
4     Decision Tree (max_depth=5)  0.017178  0.314300
3                    SVR (Linear)  0.017553  0.299331
2                       SVR (RBF)  0.019677  0.214554
5  Decision Tree (max_depth=None)  0.024604  0.017889
0                      k-NN (k=3)  0.026832 -0.071044
1                      k-NN (k=5)  0.026963 -0.076287


### **c. Utilice hiperparámetros estándar, la búsqueda en cuadrícula y búsqueda aleatoria para poder  encontrar el modelo de clasificación con mejor desempeño. Luego, obtenga el mejor accuracy  junto con los hiperparámetros óptimos para dicho modelo.**

In [12]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [16]:
# Convertir y_final a DataFrame con nombre correcto
y_final = pd.DataFrame(y_final)
y_final.columns = ['actual_productivity']

# Crear variable categórica
y_class = pd.cut(
    y_final['actual_productivity'],
    bins=[0, 0.5, 0.75, 1.0],
    labels=['Baja', 'Media', 'Alta']
)

In [17]:
# Crear variable categórica a partir de la productividad real
y_class = pd.cut(
    y_final['actual_productivity'],
    bins=[0, 0.5, 0.75, 1.0],
    labels=['Baja', 'Media', 'Alta']
)


In [19]:
# Asegurar que y_final sea DataFrame con la columna correcta
y_final = pd.DataFrame(y_final)
y_final.columns = ['actual_productivity']

# Verificar si hay valores nulos
print(y_final['actual_productivity'].isnull().sum())

# Si hay nulos, eliminarlos (o puedes imputar si prefieres)
y_final = y_final.dropna()

# Crear variable categórica (como string)
y_class = pd.cut(
    y_final['actual_productivity'],
    bins=[0, 0.5, 0.75, 1.0],
    labels=['Baja', 'Media', 'Alta']
).astype(str)


0


In [20]:
# Asegurar que X_scaled tenga las mismas filas que y_final después del dropna
X_scaled = pd.DataFrame(X_scaled)
X_scaled = X_scaled.loc[y_final.index]


In [21]:
from sklearn.model_selection import train_test_split

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class
)


In [22]:
# Dividir datos de nuevo con la nueva y_class
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_scaled, y_class, test_size=0.2, random_state=42)

# Modelos y búsquedas
modelos = {
    "LogisticRegression": {
        "modelo": LogisticRegression(max_iter=1000),
        "param_grid": {
            "C": [0.1, 1, 10],
            "solver": ["liblinear", "lbfgs"]
        }
    },
    "RandomForest": {
        "modelo": RandomForestClassifier(),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5]
        }
    },
    "DecisionTree": {
        "modelo": DecisionTreeClassifier(),
        "param_grid": {
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 4]
        }
    },
    "SVC": {
        "modelo": SVC(),
        "param_grid": {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf"]
        }
    }
}

mejor_modelo = None
mejor_score = 0
mejor_nombre = ""
mejor_params = {}

for nombre, info in modelos.items():
    print(f"\n🔍 Buscando el mejor {nombre} con GridSearchCV...")
    grid = GridSearchCV(info["modelo"], info["param_grid"], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_cls, y_train_cls)

    y_pred = grid.predict(X_test_cls)
    acc = accuracy_score(y_test_cls, y_pred)
    print(f"Accuracy en test: {acc:.4f}")
    print("Mejores hiperparámetros:", grid.best_params_)

    if acc > mejor_score:
        mejor_modelo = grid.best_estimator_
        mejor_score = acc
        mejor_nombre = nombre
        mejor_params = grid.best_params_


🔍 Buscando el mejor LogisticRegression con GridSearchCV...
Accuracy en test: 0.6625
Mejores hiperparámetros: {'C': 1, 'solver': 'lbfgs'}

🔍 Buscando el mejor RandomForest con GridSearchCV...
Accuracy en test: 0.7792
Mejores hiperparámetros: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}

🔍 Buscando el mejor DecisionTree con GridSearchCV...
Accuracy en test: 0.7667
Mejores hiperparámetros: {'max_depth': 5, 'min_samples_split': 4}

🔍 Buscando el mejor SVC con GridSearchCV...
Accuracy en test: 0.6708
Mejores hiperparámetros: {'C': 1, 'kernel': 'linear'}


In [23]:
# Reporte final
print("\n✅ Mejor modelo encontrado:")
print("Modelo:", mejor_nombre)
print("Accuracy:", mejor_score)
print("Mejores hiperparámetros:", mejor_params)

# Informe detallado
print("\n📋 Classification Report del mejor modelo:")
y_pred_mejor = mejor_modelo.predict(X_test_cls)
print(classification_report(y_test_cls, y_pred_mejor))


✅ Mejor modelo encontrado:
Modelo: RandomForest
Accuracy: 0.7791666666666667
Mejores hiperparámetros: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}

📋 Classification Report del mejor modelo:
              precision    recall  f1-score   support

        Alta       0.82      0.92      0.86       132
        Baja       0.60      0.27      0.38        22
       Media       0.71      0.69      0.70        77
         nan       1.00      0.78      0.88         9

    accuracy                           0.78       240
   macro avg       0.78      0.66      0.70       240
weighted avg       0.77      0.78      0.77       240

